# 1. Initializations

## 1.1 General imports

In [ ]:
### global
import logging
from smartcheck.logger_config import setup_logger
setup_logger(logging.INFO)
from typing import cast

### machine learning (scikit-learn)
import pandas as pd
import numpy as np
import torch
from tsfm_public import (
    TimeSeriesForecastingPipeline,
    TinyTimeMixerForPrediction,
)
from tsfm_public.toolkit.visualization import plot_predictions
from sklearn.pipeline import Pipeline
from torch.utils.data import DataLoader

### graphical
import matplotlib.pyplot as plt
# for jupyter notebook management
%matplotlib inline


## 1.2 Project Specific imports

In [ ]:

import smartcheck.dataframe_common as dfc
import smartcheck.dataframe_project_specific as dfps
import smartcheck.preprocessing_project_specific as pps
import smartcheck.deep_learning_project_specific as mps

# 2. Loading and Preprocessing

In [ ]:
df_cpt_raw = dfc.load_dataset_from_config('velo_comptage_ml_ready_data', sep=',', index_col=0)

if df_cpt_raw is not None and isinstance(df_cpt_raw, pd.DataFrame):
    df_cpt = df_cpt_raw.copy()

In [ ]:
df_cpt.info()

## 2.1 Preprocessing pipelines

In [ ]:
keep_cols = [
    "nom_du_site_de_comptage",
    "comptage_horaire",
    # "date_et_heure_de_comptage",
    "date_et_heure_de_comptage_local",
    # "date_et_heure_de_comptage_utc",
    "orientation_compteur",
    # "latitude",
    # "longitude",
    # "arrondissement",
    "jour_ferie",
    "vacances_scolaires",
    "temperature_2m_c",
    "rain_mm",
    "snowfall_cm",
    # "weather_code_wmo_code",
    # "elevation",
    "weather_code_wmo_code_category",
]

pipe_preproc = Pipeline([
    ("add_datetime_features", pps.DatetimePreprocessingTransformer(timestamp_col="date_et_heure_de_comptage",
                                                                   for_sarimax=True)),
    ("filter_columns", pps.ColumnFilterTransformer(columns_to_keep=keep_cols)),
])

df_preproc = pipe_preproc.fit_transform(df_cpt)
if df_preproc is not None and isinstance(df_preproc, pd.DataFrame):
    df = df_preproc.copy()

In [ ]:
# Verification des distributions après preprocessing
df.info()
display(df.select_dtypes(include=np.number).describe())
display(df.select_dtypes(include='object').describe())

# 3. Regression modeling

In [ ]:
dict_compteurs = {
    "experiment_1": {
        "key": ('135 avenue Daumesnil','SE-NO'),
        "name": "Daumesnil S-N",
        "sub_range": (-1000,),
    },
}
timestamp_column = "date_et_heure_de_comptage_local"
target_columns = ["comptage_horaire"]
context_length = 512
grouped_df = df.groupby(["nom_du_site_de_comptage", "orientation_compteur"])

In [ ]:
for experiment, exp_params in dict_compteurs.items():
    key = exp_params["key"]
    if key in grouped_df.groups:
        df_compteur = grouped_df.get_group(key)
        logging.info(f"\n--- {experiment} : {exp_params} ---")
    else:
        logging.info(f"⚠️ Clé {key} non trouvée dans les groupes de df.")
        continue

    df_compteur = df_compteur.sort_values(by=timestamp_column)
    sub_range = exp_params["sub_range"]
    range_start = sub_range[0]
    range_end = sub_range[1] if len(sub_range) > 1 else None
    df_compteur_sub = df_compteur[range_start:range_end]

    fig, axs = plt.subplots(len(target_columns), 1, figsize=(10, 2 * len(target_columns)), squeeze=False)
    for ax, target_column in zip(axs, target_columns):
        ax[0].plot(df_compteur_sub[timestamp_column], df_compteur_sub[target_column])
    plt.show()